# 밴드갭 실습

**Band Gap · 띠틈**

가전자대의 꼭대기와 전도대의 바닥 사이 에너지 차이.

소재 분야에서 이해하기: 반도체 후보의 전자적 성질을 비교하는 지표로 사용한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 띠 구조에서 밴드갭이 생기는 이유

원자 두 종류가 번갈아 놓인 1차원 사슬(타이트바인딩)에서 띠가 갈라지는 것을 계산합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def bands(onsite_a, onsite_b, hopping=1.0, points=400):
    """두 원자 기저 1차원 사슬의 두 밴드. 단위: eV"""
    k = np.linspace(-np.pi, np.pi, points)
    delta = (onsite_a - onsite_b) / 2
    centre = (onsite_a + onsite_b) / 2
    dispersion = np.sqrt(delta ** 2 + 4 * hopping ** 2 * np.cos(k / 2) ** 2)
    return k, centre - dispersion, centre + dispersion

for difference in (0.0, 1.0, 3.0):
    k, lower, upper = bands(difference / 2, -difference / 2)
    gap = upper.min() - lower.max()
    plt.plot(k, lower, label='onsite diff %.1f eV' % difference)
    plt.plot(k, upper, color=plt.gca().lines[-1].get_color())
    print('원자 에너지 차이 %.1f eV -> 밴드갭 %.3f eV' % (difference, max(gap, 0)))
plt.xlabel('k'); plt.ylabel('energy (eV)'); plt.legend(fontsize=8); plt.show()

## 2. 상태밀도와 밴드갭

In [ ]:
k, lower, upper = bands(1.5, -1.5)
energies = np.concatenate([lower, upper])
counts, edges = np.histogram(energies, bins=120)
centres = 0.5 * (edges[1:] + edges[:-1])
plt.plot(centres, counts)
plt.xlabel('energy (eV)'); plt.ylabel('density of states (a.u.)'); plt.show()
empty = centres[(counts == 0)]
if empty.size:
    print('상태가 없는 에너지 구간 %.2f ~ %.2f eV -> 이 폭이 밴드갭입니다' % (empty.min(), empty.max()))
print('\n금속은 이 구간이 없고, 반도체·절연체는 존재합니다. 태양전지 흡수층은 대략 1.1-1.8 eV 를 봅니다.')

## 3. 해석

밴드갭은 계산 방법에 크게 의존합니다. 표준적인 DFT(LDA/PBE)는 실험값보다 작게 나오는 경향이
알려져 있어, 데이터베이스 값을 쓸 때는 어떤 방법으로 얻은 값인지 확인해야 합니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#band-gap)을 여세요.